# 머신러닝 모델링 및 평가

전처리된 데이터를 사용하여 여러 머신러닝 모델을 학습하고 성능을 비교합니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
import joblib
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

In [ ]:
# 전처리된 데이터 로드
df = pd.read_csv('data/processed_data.csv')
print(f'데이터 크기: {df.shape}')
df.head()

In [ ]:
# 특성과 타겟 변수 분리 (실제 데이터에 맞게 수정 필요)
target_col = 'target'  # 실제 타겟 컬럼명으로 변경
if target_col in df.columns:
    X = df.drop([target_col], axis=1)
    y = df[target_col]
else:
    # 더미 타겟 생성
    numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_features) > 0:
        X = df[numeric_features].iloc[:, :-1]
        median_val = df[numeric_features[0]].median()
        y = (df[numeric_features[0]] > median_val).astype(int)
        print(f'더미 타겟 생성: {numeric_features[0]}의 중앙값 기준 이진 분류')

print(f'특성 수: {X.shape[1]}')
print(f'샘플 수: {X.shape[0]}')
print(f'타겟 분포:\n{y.value_counts()}')

In [ ]:
# 학습/테스트 세트 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'학습 세트: {X_train.shape[0]}개')
print(f'테스트 세트: {X_test.shape[0]}개')

In [ ]:
# 여러 모델 학습 및 평가
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

results = {}
for name, model in models.items():
    print(f'\n{name} 학습 중...')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    accuracy = accuracy_score(y_test, y_pred)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = {'model': model, 'accuracy': accuracy, 'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std(), 'y_pred': y_pred, 'y_pred_proba': y_pred_proba}
    print(f'테스트 정확도: {accuracy:.4f}')
    print(f'교차 검증 정확도: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})')

In [ ]:
# 모델 성능 비교
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Test Accuracy': [results[m]['accuracy'] for m in results.keys()],
    'CV Mean': [results[m]['cv_mean'] for m in results.keys()],
    'CV Std': [results[m]['cv_std'] for m in results.keys()]
}).sort_values('Test Accuracy', ascending=False)

print('모델 성능 비교:')
print(comparison_df)

plt.figure(figsize=(10, 6))
plt.barh(comparison_df['Model'], comparison_df['Test Accuracy'])
plt.xlabel('정확도')
plt.title('모델별 테스트 정확도')
plt.xlim([0, 1])
plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 최고 성능 모델 저장
best_model_name = comparison_df.iloc[0]['Model']
best_model = results[best_model_name]['model']
model_path = f'models/best_model_{best_model_name.replace(" ", "_").lower()}.pkl'
joblib.dump(best_model, model_path)
print(f'최고 성능 모델 ({best_model_name})이 {model_path}에 저장되었습니다.')